<a href="https://colab.research.google.com/github/ananthurajeev/Tracker/blob/main/Urban_Waste_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEO5017 — Urban Waste Detection Pipeline
**EfficientNet-B0 Classifier → Predict on unlabeled 4k → Top-100 submission**

### Notebook structure
1. Mount Google Drive & install libraries
2. Load & explore the CSV labels
3. Build the image path mapping (handles subfolders)
4. Train / Val / Test split (stratified)
5. Dataset class + augmentations
6. Train EfficientNet-B0 — **20 epochs, early stopping (patience=5), best model saved on waste F1**
7. Evaluate on test set — confusion matrix, ROC curve, Precision-Recall curve, threshold analysis
8. Predict on unlabeled 4,000 images
9. Export Top-100 detections


## STEP 0 — Mount Google Drive
Upload your image folder and CSV to Google Drive first, then mount it here.

In [ ]:
import os
from google.colab import drive

# Remove the stuck mount point and remount cleanly
os.system('fusermount -u /content/drive 2>/dev/null || true')
os.system('rm -rf /content/drive')

drive.mount('/content/drive')

## STEP 1 — Install libraries & set paths
▶ **Edit the two paths below to match where you uploaded things in your Drive.**

In [ ]:
!pip install -q timm scikit-learn torchvision tqdm

import os

# ── EDIT THESE TWO LINES ──────────────────────────────────────────────────────
CSV_PATH    = '/content/drive/MyDrive/GEO5017/label.csv'
IMAGES_ROOT = '/content/drive/MyDrive/GEO5017/UrbanWaste-images-10k-right'
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = '/content/drive/MyDrive/GEO5017/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Paths set. Drive mounted.')

## STEP 2 — Load & explore CSV labels

This step loads your Label Studio export and shows the class distribution.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(CSV_PATH)
print(f'Total labeled images: {len(df)}')
print()
print('Label distribution:')
print(df['choice'].value_counts())

# Plot
df['choice'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Label distribution in 6,000 annotated images')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/label_distribution.png')
plt.show()

## STEP 3 — Map filenames to actual disk paths

Your images live in subfolders like `year_2023/TMX.../filename.jpg`.  
We walk the entire image root once and build a lookup dictionary.

In [ ]:
import os
import pandas as pd
from pathlib import Path

print('Building path index from CSV full_path column...')

def windows_to_colab(win_path):
    # Extract everything after 'UrbanWaste-images-10k-right\'
    parts = win_path.replace('\\', '/').split('UrbanWaste-images-10k-right/')
    if len(parts) > 1:
        return os.path.join(IMAGES_ROOT, parts[1])
    return None

df['disk_path'] = df['full_path'].apply(windows_to_colab)

# Verify a few exist
sample = df['disk_path'].dropna().iloc[0]
print(f"Sample path: {sample}")
print(f"Sample exists: {os.path.exists(sample)}")

missing = df['disk_path'].isna().sum()
print(f"\nLabeled images mapped: {len(df) - missing} / {len(df)}")

# Unlabeled = all 10k minus the 6k labeled ones
labeled_filenames = set(df['filename'].tolist())
all_filenames = set()
for p in Path(IMAGES_ROOT).rglob('*.jpg'):
    all_filenames.add(p.name)
unlabeled_filenames = all_filenames - labeled_filenames
print(f"Unlabeled images: {len(unlabeled_filenames)}")

## STEP 4 — Create binary + multiclass labels

We create two label columns:
- `binary_label`: 0 = clean, 1 = waste (for classifier training)
- `waste_category`: the original fine-grained label (for bonus task later)

In [ ]:
# Normalize labels
df['choice'] = df['choice'].str.strip().str.lower()

# Binary label
df['binary_label'] = (df['choice'] != 'clean').astype(int)

# Fine-grained category (for bonus)
CATEGORY_MAP = {
    'clean': 0,
    'litter': 1,
    'bulky_waste': 2,
    'garbage_bag': 3,
    'waste bin': 4,   # label studio exported as 'Waste Bin'
}
df['category_label'] = df['choice'].map(CATEGORY_MAP).fillna(0).astype(int)

print('Binary labels:')
print(df['binary_label'].value_counts())
print()
print('Category labels:')
print(df['category_label'].value_counts())

## STEP 5 — Train / Val / Test split (stratified)

In [ ]:
from sklearn.model_selection import train_test_split

# 70% train, 15% val, 15% test — stratified by binary label
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['binary_label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['binary_label'], random_state=42
)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print()
print('Waste % in each split:')
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    pct = split['binary_label'].mean() * 100
    print(f'  {name}: {pct:.1f}% waste')

## STEP 6 — PyTorch Dataset & DataLoaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# Transforms
TRAIN_TRANSFORMS = transforms.Compose([
    transforms.Resize((300, 300)),   # ← B3 native size
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

VAL_TRANSFORMS = transforms.Compose([
    transforms.Resize((300, 300)),   # ← match train size
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class WasteDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['disk_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row['binary_label'], dtype=torch.long)
        return img, label, row['filename']

# DataLoaders
BATCH_SIZE = 32
train_dataset = WasteDataset(train_df, TRAIN_TRANSFORMS)
val_dataset   = WasteDataset(val_df,   VAL_TRANSFORMS)
test_dataset  = WasteDataset(test_df,  VAL_TRANSFORMS)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## STEP 7 — Build model (EfficientNet-B0) with class weights

We use **class weights** to handle the severe imbalance (5449 clean vs 528 waste).

In [ ]:
# ── STEP 7 — CLIP-based classifier (completely different approach) ──
!pip install -q open_clip_torch

import open_clip
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import random

# Add set_seed here
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f'Seed set: {seed}')

set_seed(42)

# Load CLIP model
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
clip_model = clip_model.to(device)

# Freeze CLIP backbone — only train a small classifier on top
for param in clip_model.parameters():
    param.requires_grad = False

# Small classifier head on top of CLIP features
class CLIPClassifier(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.clip = clip_model
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        with torch.no_grad():
            features = self.clip.encode_image(x)
            features = features.float()
        return self.classifier(features)

model = CLIPClassifier(clip_model).to(device)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

# ── CLIP needs its own preprocessing ──
from torchvision import transforms
from PIL import Image

# Override transforms with CLIP's own preprocessing
CLIP_TRANSFORM = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.48145466, 0.4578275, 0.40821073),
        std=(0.26862954, 0.26130258, 0.27577711)
    )
])

class WasteDatasetCLIP(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['disk_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row['binary_label'], dtype=torch.long)
        return img, label, row['filename']

# Rebuild datasets with CLIP transforms
train_dataset_clip = WasteDatasetCLIP(train_df, CLIP_TRANSFORM)
val_dataset_clip   = WasteDatasetCLIP(val_df,   CLIP_TRANSFORM)
test_dataset_clip  = WasteDatasetCLIP(test_df,  CLIP_TRANSFORM)

train_loader_clip = DataLoader(train_dataset_clip, batch_size=64, shuffle=True,  num_workers=2)
val_loader_clip   = DataLoader(val_dataset_clip,   batch_size=64, shuffle=False, num_workers=2)
test_loader_clip  = DataLoader(test_dataset_clip,  batch_size=64, shuffle=False, num_workers=2)

# Class weights
train_labels = train_df['binary_label'].values
class_weights = compute_class_weight('balanced', classes=np.array([0,1]), y=train_labels)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

# Only train the classifier head — very fast
optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print('CLIP classifier ready.')

## STEP 8 — Training loop

Key improvements vs a naive setup:
- **20 epochs** — gives the model more passes to learn the minority waste class
- **Early stopping** (patience=5) — stops automatically if val waste F1 doesn't improve for 5 epochs in a row
- **Best model saved on val waste F1** (not val loss) — critical for imbalanced data; a model predicting 'clean' for everything gets low loss but is useless
- **Per-epoch print** shows Precision, Recall, and F1 on the waste class so you can monitor learning progress

In [ ]:
# ── STEP 8 — Train CLIP classifier ──
from sklearn.metrics import f1_score, precision_score, recall_score
import torch.nn.functional as F

NUM_EPOCHS = 10
PATIENCE = 5
best_val_f1 = 0.0
patience_counter = 0
MODEL_SAVE_PATH = f'{OUTPUT_DIR}/best_model_clip.pth'

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for imgs, labels, _ in tqdm(train_loader_clip, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}'):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for imgs, labels, _ in val_loader_clip:
            imgs = imgs.to(device)
            out = model(imgs)
            all_probs.extend(F.softmax(out, dim=1)[:,1].cpu().numpy())
            all_labels.extend(labels.numpy())

    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    # Threshold sweep
    best_f1, best_thresh = 0, 0.5
    for thresh in np.arange(0.1, 0.95, 0.05):
        preds = (all_probs >= thresh).astype(int)
        f = f1_score(all_labels, preds, pos_label=1, zero_division=0)
        if f > best_f1:
            best_f1, best_thresh = f, thresh

    preds = (all_probs >= best_thresh).astype(int)
    prec = precision_score(all_labels, preds, pos_label=1, zero_division=0)
    rec  = recall_score(all_labels, preds, pos_label=1, zero_division=0)
    scheduler.step()

    print(f'Epoch {epoch+1:02d} | Loss: {running_loss/len(train_loader_clip):.4f} | '
          f'Thresh: {best_thresh:.2f} | Prec: {prec:.3f} Rec: {rec:.3f} F1: {best_f1:.3f}')

    if best_f1 > best_val_f1:
        best_val_f1 = best_f1
        BEST_THRESHOLD = best_thresh
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f'  ✅ Saved (F1={best_val_f1:.4f} @ thresh={best_thresh:.2f})')
    else:
        patience_counter += 1
        print(f'  ⏳ No improvement {patience_counter}/{PATIENCE}')
        if patience_counter >= PATIENCE:
            print('🛑 Early stopping')
            break

print(f'\n✅ Best CLIP F1: {best_val_f1:.4f}')

## STEP 9 — Evaluate on Test Set

Loads the best saved model and runs it on the held-out test set (never seen during training).
Outputs a classification report (precision, recall, F1 per class) and a confusion matrix.

In [ ]:
# ── STEP 9 — Evaluate on Test Set (CLIP model) ──
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import torch.nn.functional as F
import numpy as np

# Load best model
model.load_state_dict(torch.load(f'{OUTPUT_DIR}/best_model_clip.pth'))
model.eval()

all_labels, all_preds, all_probs = [], [], []

with torch.no_grad():
    for imgs, labels, _ in tqdm(test_loader_clip, desc='Evaluating on test set'):
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = F.softmax(outputs, dim=1)[:, 1]
        all_labels.extend(labels.numpy())
        all_preds.extend((probs.cpu().numpy() >= BEST_THRESHOLD).astype(int))
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

print(f'Using threshold: {BEST_THRESHOLD:.2f}')
print()
print('Classification Report:')
print(classification_report(all_labels, all_preds, target_names=['clean', 'waste']))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['clean', 'waste'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Test Set (CLIP)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix_clip.png')
plt.show()

# Probability distribution plot
waste_probs = all_probs[all_labels == 1]
clean_probs = all_probs[all_labels == 0]
plt.figure(figsize=(10, 4))
plt.hist(clean_probs, bins=50, alpha=0.6, label='Clean', color='blue')
plt.hist(waste_probs, bins=50, alpha=0.6, label='Waste', color='red')
plt.axvline(BEST_THRESHOLD, color='black', linestyle='--', label=f'Threshold={BEST_THRESHOLD:.2f}')
plt.xlabel('Predicted waste probability')
plt.ylabel('Count')
plt.title('Probability Distribution — CLIP model')
plt.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/prob_distribution_clip.png')
plt.show()

## STEP 10 — Predict on Unlabeled 4,000 Images

In [ ]:
# ── STEP 10 — Predict on unlabeled 4,000 (CLIP model) ──
from pathlib import Path

class UnlabeledDatasetCLIP(Dataset):
    def __init__(self, filenames, filename_to_path, transform):
        self.filenames = list(filenames)
        self.filename_to_path = filename_to_path
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img = Image.open(self.filename_to_path[fname]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, fname

# Build filename_to_path for unlabeled images
unlabeled_paths = {}
for fname in unlabeled_filenames:
    # reconstruct path from IMAGES_ROOT
    for p in Path(IMAGES_ROOT).rglob(fname):
        unlabeled_paths[fname] = str(p)
        break

unlabeled_dataset_clip = UnlabeledDatasetCLIP(
    unlabeled_filenames, unlabeled_paths, CLIP_TRANSFORM
)
unlabeled_loader_clip = DataLoader(
    unlabeled_dataset_clip, batch_size=64, shuffle=False, num_workers=2
)

model.eval()
unlabeled_results = []

with torch.no_grad():
    for imgs, fnames in tqdm(unlabeled_loader_clip, desc='Predicting on unlabeled images'):
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = F.softmax(outputs, dim=1)[:, 1]
        for fname, prob in zip(fnames, probs.cpu().numpy()):
            unlabeled_results.append({
                'filename': fname,
                'waste_prob': float(prob)
            })

results_df = pd.DataFrame(unlabeled_results).sort_values('waste_prob', ascending=False)
print(f'Predictions done.')
print(f'Top 5:')
print(results_df.head())
print(f'\nProb range: {results_df["waste_prob"].min():.3f} – {results_df["waste_prob"].max():.3f}')

## STEP 11 — Export Top-100 for Leaderboard Submission

In [ ]:
import shutil

TOP100_DIR = f'{OUTPUT_DIR}/top100_waste_detections'
os.makedirs(TOP100_DIR, exist_ok=True)

top100 = results_df.head(100)

# Copy images
for _, row in tqdm(top100.iterrows(), total=100, desc='Copying Top-100 images'):
    src = filename_to_path[row['filename']]
    dst = os.path.join(TOP100_DIR, row['filename'])
    shutil.copy2(src, dst)

# Save ranked list with confidence scores
top100.to_csv(f'{OUTPUT_DIR}/top100_ranked.csv', index=False)

print(f'\n✅ Top-100 images saved to: {TOP100_DIR}')
print(f'✅ Ranked CSV saved to: {OUTPUT_DIR}/top100_ranked.csv')
print(f'\nTop-100 waste probability range: {top100["waste_prob"].min():.3f} – {top100["waste_prob"].max():.3f}')

## STEP 12 — Visualize some Top-100 predictions

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for ax, (_, row) in zip(axes.flatten(), top100.head(10).iterrows()):
    img = Image.open(filename_to_path[row['filename']]).convert('RGB')
    ax.imshow(img)
    ax.set_title(f"p={row['waste_prob']:.3f}", fontsize=9)
    ax.axis('off')
plt.suptitle('Top-10 highest confidence waste detections', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/top10_preview.png')
plt.show()

---
## BONUS — YOLOv8 for Localization + Classification

> ⚠️ **Before running this section**, you need to go back to Label Studio and draw **bounding boxes** with category labels on your waste-positive images (~528 images). Export in YOLO format.

### Why YOLO?
- Detects AND localizes waste in a single forward pass
- Supports multi-class output (litter, bulky_waste, garbage_bag, waste_bin)
- Easy to run on Colab with GPU


In [ ]:
# Install YOLOv8
!pip install -q ultralytics

In [ ]:
# ── EDIT: point to your YOLO-format dataset exported from Label Studio ──
YOLO_DATA_YAML = '/content/drive/MyDrive/GEO5017/yolo_dataset/data.yaml'

from ultralytics import YOLO

# Load pretrained YOLOv8 small
yolo_model = YOLO('yolov8s.pt')

# Train
results = yolo_model.train(
    data=YOLO_DATA_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    project=OUTPUT_DIR,
    name='yolo_waste',
    patience=10,   # early stopping
    device=0       # GPU
)

In [ ]:
# Run YOLO inference on the 4,000 unlabeled images to get localized detections
YOLO_MODEL_PATH = f'{OUTPUT_DIR}/yolo_waste/weights/best.pt'
yolo_model = YOLO(YOLO_MODEL_PATH)

unlabeled_paths = [filename_to_path[f] for f in unlabeled_filenames]

yolo_results = []
for path in tqdm(unlabeled_paths, desc='YOLO inference'):
    res = yolo_model(path, verbose=False)[0]
    if len(res.boxes) > 0:
        max_conf = float(res.boxes.conf.max())
    else:
        max_conf = 0.0
    yolo_results.append({'filename': os.path.basename(path), 'yolo_conf': max_conf, 'path': path})

yolo_df = pd.DataFrame(yolo_results).sort_values('yolo_conf', ascending=False)
print('Top 5 YOLO detections:')
print(yolo_df.head())

In [ ]:
# Save YOLO Top-100 with annotated bounding boxes
YOLO_TOP100_DIR = f'{OUTPUT_DIR}/yolo_top100'
os.makedirs(YOLO_TOP100_DIR, exist_ok=True)

for _, row in tqdm(yolo_df.head(100).iterrows(), total=100, desc='Saving YOLO Top-100'):
    res = yolo_model(row['path'], verbose=False)[0]
    annotated = res.plot()  # draws bounding boxes on image
    save_path = os.path.join(YOLO_TOP100_DIR, row['filename'])
    Image.fromarray(annotated).save(save_path)

print(f'\n✅ YOLO Top-100 with bounding boxes saved to: {YOLO_TOP100_DIR}')